In [ ]:
from datasets import Dataset, load_dataset

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
dataset = dataset.filter(lambda x: x["category"] == "ACCOUNT")
dataset = dataset.shuffle(seed=42)
# dataset[0]

In [ ]:
import dspy

train_samples = dataset.select(range(1000))
dev_samples = dataset.select(range(1000, 1100))

trainset = [dspy.Example(instruction=ex['instruction'], intent=ex['intent']).with_inputs('instruction') for ex in train_samples]
devset = [dspy.Example(instruction=ex['instruction'], intent=ex['intent']).with_inputs('instruction') for ex in dev_samples]

print(f"Anzahl der Beispiele im Trainingsset: {len(trainset)}")
print(f"Anzahl der Beispiele im Evaluationsset: {len(devset)}")

# Check 5 elements of the trainset
# print(trainset[:5])

In [ ]:
# Prüfen, ob alle Intents in train und dev vorhanden sind
train_intents = list(train_samples.unique("intent"))
print(train_intents)
dev_intents = list(dev_samples.unique("intent"))
print(dev_intents)

In [ ]:
class IntentSignature(dspy.Signature):
    """
    Identify the intent of the customer instruction. 
    Possible values are recover_password, switch_account, create_account, delete_account, registration_problems, edit_account.
    """
    instruction = dspy.InputField(desc="Instruction: a user request from the Customer Service domain.")
    intent = dspy.OutputField(desc="Intent: the intent corresponding to the user instruction.")

class IntentClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(IntentSignature)

    def forward(self, instruction):
        return self.predictor(instruction=instruction)

In [ ]:
# Konfiguration des lokalen Sprachmodells
local_llm = dspy.LM(
    "openai/gemma-3-4b-it-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=0.1,
    cache=False
)

dspy.configure(lm=local_llm)

In [ ]:
from dspy.evaluate import Evaluate

# Metrik-Funktion: Vergleicht Vorhersage und Label (case-insensitive)
def exact_match_metric(gold, pred, trace=None):
    return gold.intent.lower() == pred.intent.lower()

# Evaluator instanziieren
evaluator = Evaluate(devset=devset, metric=exact_match_metric, num_threads=1, display_progress=True)

# Test des unoptimierten Modells (Zero-Shot)
print("Evaluation vor Optimierung:")
unoptimized_program = IntentClassifier()
evaluator(unoptimized_program, display_table=0)

In [ ]:
from dspy.teleprompt import BootstrapFewShot

# Konfiguration des Optimizers
# max_bootstrapped_demos: Maximale Anzahl an generierten Beispielen im Prompt
optimizer = BootstrapFewShot(metric=exact_match_metric, max_bootstrapped_demos=4)

# Kompilierung (Optimierung) des Programms mit den Trainingsdaten
print("\nStarte Optimierung...")
optimized_program = optimizer.compile(IntentClassifier(), trainset=trainset)

In [ ]:
print("\nEvaluation nach Optimierung:")
evaluator(optimized_program, display_table=0)

In [ ]:
# Optional: Inspektion des optimierten Prompts (zeigt die ausgewählten Beispiele)
dspy.settings.lm.inspect_history(n=1)

In [ ]:
# Listen zum Speichern von Labels und Vorhersagen
y_true = []
y_pred = []

# Metric-Funktion so anpassen, dass wir die Werte sammeln
def exact_match_metric(gold, pred, trace=None):
    y_true.append(gold.intent)
    y_pred.append(pred.intent)
    return gold.intent.lower() == pred.intent.lower()

In [ ]:
from dspy.evaluate import Evaluate
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Evaluator instanziieren
evaluator = Evaluate(devset=devset, metric=exact_match_metric, num_threads=1, display_progress=True)

# Evaluation starten
results = evaluator(optimized_program, display_table=0)

# Confusion-Matrix erzeugen
labels = ['recover_password', 'switch_account', 'create_account', 
          'delete_account', 'registration_problems', 'edit_account']
cm = confusion_matrix(y_true, y_pred, labels=labels)

# Confusion-Matrix anzeigen
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.show()